In [ ]:
# PHASE 7: HYBRID DETECTION SYSTEM INTEGRATION
# TASK 1: FUSION STRATEGY DEVELOPMENT
# CNN + Rule-based combination algorithms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import os
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PHASE 7 - TASK 1: FUSION STRATEGY DEVELOPMENT")
print("="*80)
print("Task: CNN + Rule-based combination algorithms")
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print()

class FusionStrategyEngine:
    """
    Advanced Fusion Strategy Engine for combining CNN and Rule-based predictions
    Implements multiple fusion algorithms for hybrid SQL injection detection
    """
    
    def __init__(self):
        self.fusion_strategies = {}
        self.performance_metrics = {}
        self.confidence_thresholds = {
            'high_confidence': 0.8,
            'medium_confidence': 0.6,
            'low_confidence': 0.4
        }
        self.initialize_fusion_strategies()
        
    def initialize_fusion_strategies(self):
        """Initialize different fusion strategy implementations"""
        
        print("1. INITIALIZING FUSION STRATEGIES:")
        print("-" * 50)
        
        # Strategy 1: Weighted Average Fusion
        self.fusion_strategies['weighted_average'] = {
            'name': 'Weighted Average Fusion',
            'description': 'Combines predictions using learned weights',
            'weights': {'cnn': 0.7, 'rule_based': 0.3},  # CNN has higher weight due to 99.51% vs 84.86%
            'active': True
        }
        
        # Strategy 2: Confidence-Based Dynamic Fusion
        self.fusion_strategies['confidence_dynamic'] = {
            'name': 'Confidence-Based Dynamic Fusion',
            'description': 'Adjusts weights based on prediction confidence',
            'base_weights': {'cnn': 0.65, 'rule_based': 0.35},
            'confidence_boost': 0.2,
            'active': True
        }
        
        # Strategy 3: Voting-Based Fusion
        self.fusion_strategies['majority_voting'] = {
            'name': 'Majority Voting',
            'description': 'Binary decision based on majority vote',
            'threshold': 0.5,
            'active': True
        }
        
        # Strategy 4: Hierarchical Fusion
        self.fusion_strategies['hierarchical'] = {
            'name': 'Hierarchical Decision Fusion',
            'description': 'CNN first, rule-based for edge cases',
            'cnn_confidence_threshold': 0.7,
            'active': True
        }
        
        # Strategy 5: Ensemble Fusion
        self.fusion_strategies['ensemble'] = {
            'name': 'Advanced Ensemble Fusion',
            'description': 'Multiple fusion methods with meta-learning',
            'meta_weights': [0.4, 0.3, 0.2, 0.1],  # Weights for different fusion outputs
            'active': True
        }
        
        for strategy_key, strategy in self.fusion_strategies.items():
            status = "ACTIVE" if strategy['active'] else "INACTIVE"
            print(f"  Strategy: {strategy['name']:<30} [{status}]")
            print(f"           {strategy['description']}")
        
        print(f"\nTotal Fusion Strategies: {len(self.fusion_strategies)}")
        
    def weighted_average_fusion(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """
        Strategy 1: Weighted Average Fusion
        Combines predictions using predefined weights based on historical performance
        """
        weights = self.fusion_strategies['weighted_average']['weights']
        
        # Basic weighted average
        fused_score = (cnn_pred * weights['cnn']) + (rule_pred * weights['rule_based'])
        
        # Confidence calculation
        fused_confidence = (cnn_conf * weights['cnn']) + (rule_conf * weights['rule_based'])
        
        return {
            'prediction': fused_score,
            'confidence': fused_confidence,
            'strategy': 'weighted_average',
            'details': {
                'cnn_weight': weights['cnn'],
                'rule_weight': weights['rule_based'],
                'raw_cnn': cnn_pred,
                'raw_rule': rule_pred
            }
        }
    
    def confidence_dynamic_fusion(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """
        Strategy 2: Confidence-Based Dynamic Fusion
        Adjusts weights dynamically based on prediction confidence levels
        """
        base_weights = self.fusion_strategies['confidence_dynamic']['base_weights']
        boost = self.fusion_strategies['confidence_dynamic']['confidence_boost']
        
        # Calculate confidence differential
        conf_diff = abs(cnn_conf - rule_conf)
        
        # Adjust weights based on confidence
        if cnn_conf > rule_conf:
            cnn_weight = min(base_weights['cnn'] + (boost * conf_diff), 0.9)
            rule_weight = 1.0 - cnn_weight
        else:
            rule_weight = min(base_weights['rule_based'] + (boost * conf_diff), 0.9)
            cnn_weight = 1.0 - rule_weight
        
        # Fusion calculation
        fused_score = (cnn_pred * cnn_weight) + (rule_pred * rule_weight)
        fused_confidence = (cnn_conf * cnn_weight) + (rule_conf * rule_weight)
        
        return {
            'prediction': fused_score,
            'confidence': fused_confidence,
            'strategy': 'confidence_dynamic',
            'details': {
                'dynamic_cnn_weight': cnn_weight,
                'dynamic_rule_weight': rule_weight,
                'confidence_diff': conf_diff,
                'raw_cnn': cnn_pred,
                'raw_rule': rule_pred
            }
        }
    
    def majority_voting_fusion(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """
        Strategy 3: Majority Voting
        Simple binary decision based on majority agreement
        """
        threshold = self.fusion_strategies['majority_voting']['threshold']
        
        # Convert to binary decisions
        cnn_decision = 1 if cnn_pred > threshold else 0
        rule_decision = 1 if rule_pred > threshold else 0
        
        # Majority vote
        if cnn_decision == rule_decision:
            fused_prediction = cnn_decision
            fused_confidence = (cnn_conf + rule_conf) / 2
        else:
            # Tie-breaker: use higher confidence prediction
            if cnn_conf > rule_conf:
                fused_prediction = cnn_decision
                fused_confidence = cnn_conf
            else:
                fused_prediction = rule_decision
                fused_confidence = rule_conf
        
        return {
            'prediction': fused_prediction,
            'confidence': fused_confidence,
            'strategy': 'majority_voting',
            'details': {
                'cnn_decision': cnn_decision,
                'rule_decision': rule_decision,
                'agreement': cnn_decision == rule_decision,
                'tie_breaker_used': cnn_decision != rule_decision
            }
        }
    
    def hierarchical_fusion(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """
        Strategy 4: Hierarchical Decision Fusion
        CNN takes precedence, rule-based system handles edge cases
        """
        cnn_threshold = self.fusion_strategies['hierarchical']['cnn_confidence_threshold']
        
        # Primary decision: CNN with high confidence
        if cnn_conf >= cnn_threshold:
            return {
                'prediction': cnn_pred,
                'confidence': cnn_conf,
                'strategy': 'hierarchical',
                'details': {
                    'decision_maker': 'cnn_primary',
                    'cnn_confidence_sufficient': True,
                    'rule_based_consulted': False
                }
            }
        else:
            # Secondary decision: Weighted combination for uncertain cases
            uncertain_cnn_weight = 0.6
            uncertain_rule_weight = 0.4
            
            fused_score = (cnn_pred * uncertain_cnn_weight) + (rule_pred * uncertain_rule_weight)
            fused_confidence = (cnn_conf * uncertain_cnn_weight) + (rule_conf * uncertain_rule_weight)
            
            return {
                'prediction': fused_score,
                'confidence': fused_confidence,
                'strategy': 'hierarchical',
                'details': {
                    'decision_maker': 'hybrid_uncertain',
                    'cnn_confidence_sufficient': False,
                    'rule_based_consulted': True,
                    'uncertainty_weights': {
                        'cnn': uncertain_cnn_weight,
                        'rule': uncertain_rule_weight
                    }
                }
            }
    
    def ensemble_fusion(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """
        Strategy 5: Advanced Ensemble Fusion
        Combines multiple fusion strategies with meta-learning weights
        """
        # Get predictions from all other strategies
        strategy_results = []
        
        # Weighted Average
        result1 = self.weighted_average_fusion(cnn_pred, rule_pred, cnn_conf, rule_conf)
        strategy_results.append(result1['prediction'])
        
        # Confidence Dynamic
        result2 = self.confidence_dynamic_fusion(cnn_pred, rule_pred, cnn_conf, rule_conf)
        strategy_results.append(result2['prediction'])
        
        # Majority Voting
        result3 = self.majority_voting_fusion(cnn_pred, rule_pred, cnn_conf, rule_conf)
        strategy_results.append(result3['prediction'])
        
        # Hierarchical
        result4 = self.hierarchical_fusion(cnn_pred, rule_pred, cnn_conf, rule_conf)
        strategy_results.append(result4['prediction'])
        
        # Meta-learning ensemble
        meta_weights = self.fusion_strategies['ensemble']['meta_weights']
        ensemble_prediction = sum(pred * weight for pred, weight in zip(strategy_results, meta_weights))
        
        # Ensemble confidence (average of strategy confidences)
        strategy_confidences = [result1['confidence'], result2['confidence'], 
                              result3['confidence'], result4['confidence']]
        ensemble_confidence = sum(conf * weight for conf, weight in zip(strategy_confidences, meta_weights))
        
        return {
            'prediction': ensemble_prediction,
            'confidence': ensemble_confidence,
            'strategy': 'ensemble',
            'details': {
                'strategy_predictions': strategy_results,
                'strategy_confidences': strategy_confidences,
                'meta_weights': meta_weights,
                'strategy_agreement': np.std(strategy_results)  # Lower std = more agreement
            }
        }
    
    def test_fusion_strategies(self):
        """Test all fusion strategies with sample data"""
        print("\n2. TESTING FUSION STRATEGIES:")
        print("-" * 50)
        
        # Sample test cases representing different scenarios
        test_cases = [
            {
                'name': 'High CNN Confidence Case',
                'cnn_pred': 0.95, 'cnn_conf': 0.92,
                'rule_pred': 0.75, 'rule_conf': 0.68,
                'expected': 'CNN should dominate'
            },
            {
                'name': 'High Rule Confidence Case',
                'cnn_pred': 0.55, 'cnn_conf': 0.58,
                'rule_pred': 0.88, 'rule_conf': 0.91,
                'expected': 'Rule-based should influence more'
            },
            {
                'name': 'Agreement Case',
                'cnn_pred': 0.85, 'cnn_conf': 0.80,
                'rule_pred': 0.82, 'rule_conf': 0.78,
                'expected': 'Both systems agree - high confidence'
            },
            {
                'name': 'Disagreement Case',
                'cnn_pred': 0.25, 'cnn_conf': 0.65,
                'rule_pred': 0.85, 'rule_conf': 0.70,
                'expected': 'Systems disagree - complex fusion needed'
            },
            {
                'name': 'Low Confidence Case',
                'cnn_pred': 0.52, 'cnn_conf': 0.45,
                'rule_pred': 0.48, 'rule_conf': 0.42,
                'expected': 'Both uncertain - conservative approach'
            }
        ]
        
        fusion_results = []
        
        for i, test_case in enumerate(test_cases, 1):
            print(f"\nTest Case {i}: {test_case['name']}")
            print(f"CNN: pred={test_case['cnn_pred']:.3f}, conf={test_case['cnn_conf']:.3f}")
            print(f"Rule: pred={test_case['rule_pred']:.3f}, conf={test_case['rule_conf']:.3f}")
            print(f"Expected: {test_case['expected']}")
            print()
            
            case_results = {}
            
            # Test each fusion strategy
            strategies_to_test = [
                ('weighted_average', self.weighted_average_fusion),
                ('confidence_dynamic', self.confidence_dynamic_fusion),
                ('majority_voting', self.majority_voting_fusion),
                ('hierarchical', self.hierarchical_fusion),
                ('ensemble', self.ensemble_fusion)
            ]
            
            for strategy_name, strategy_func in strategies_to_test:
                result = strategy_func(
                    test_case['cnn_pred'], test_case['rule_pred'],
                    test_case['cnn_conf'], test_case['rule_conf']
                )
                case_results[strategy_name] = result
                print(f"  {strategy_name:20}: pred={result['prediction']:.3f}, conf={result['confidence']:.3f}")
            
            fusion_results.append({
                'test_case': test_case['name'],
                'input': test_case,
                'results': case_results
            })
            
            print("-" * 40)
        
        return fusion_results

# Initialize and test the fusion strategy engine
print("INITIALIZING FUSION STRATEGY ENGINE...")
fusion_engine = FusionStrategyEngine()

# Run comprehensive tests
test_results = fusion_engine.test_fusion_strategies()

print("\n3. FUSION STRATEGY ANALYSIS:")
print("-" * 50)

# Analyze strategy performance patterns
strategy_names = ['weighted_average', 'confidence_dynamic', 'majority_voting', 'hierarchical', 'ensemble']
analysis_data = {strategy: [] for strategy in strategy_names}

for test_result in test_results:
    for strategy in strategy_names:
        pred = test_result['results'][strategy]['prediction']
        conf = test_result['results'][strategy]['confidence']
        analysis_data[strategy].append({'prediction': pred, 'confidence': conf})

# Calculate strategy statistics
strategy_stats = {}
for strategy in strategy_names:
    predictions = [result['prediction'] for result in analysis_data[strategy]]
    confidences = [result['confidence'] for result in analysis_data[strategy]]
    
    strategy_stats[strategy] = {
        'avg_prediction': np.mean(predictions),
        'std_prediction': np.std(predictions),
        'avg_confidence': np.mean(confidences),
        'std_confidence': np.std(confidences),
        'prediction_range': max(predictions) - min(predictions)
    }

# Display strategy statistics
print("Strategy Performance Statistics:")
for strategy, stats in strategy_stats.items():
    print(f"\n{strategy.upper()}:")
    print(f"  Avg Prediction: {stats['avg_prediction']:.3f} (±{stats['std_prediction']:.3f})")
    print(f"  Avg Confidence: {stats['avg_confidence']:.3f} (±{stats['std_confidence']:.3f})")
    print(f"  Prediction Range: {stats['prediction_range']:.3f}")

# Create visualization
print("\n4. GENERATING FUSION STRATEGY VISUALIZATIONS:")
print("-" * 50)

plt.figure(figsize=(15, 10))

# Plot 1: Strategy Predictions Comparison
plt.subplot(2, 3, 1)
strategies = list(strategy_stats.keys())
avg_predictions = [strategy_stats[s]['avg_prediction'] for s in strategies]
plt.bar(strategies, avg_predictions, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
plt.title('Average Predictions by Strategy')
plt.ylabel('Average Prediction Score')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Plot 2: Strategy Confidence Comparison
plt.subplot(2, 3, 2)
avg_confidences = [strategy_stats[s]['avg_confidence'] for s in strategies]
plt.bar(strategies, avg_confidences, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
plt.title('Average Confidence by Strategy')
plt.ylabel('Average Confidence Score')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Plot 3: Prediction Variability
plt.subplot(2, 3, 3)
std_predictions = [strategy_stats[s]['std_prediction'] for s in strategies]
plt.bar(strategies, std_predictions, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
plt.title('Prediction Variability (Std Dev)')
plt.ylabel('Standard Deviation')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Plot 4: Strategy Decision Patterns
plt.subplot(2, 3, 4)
test_case_names = [result['test_case'] for result in test_results]
strategy_predictions_matrix = []
for strategy in strategies:
    strategy_preds = [result['results'][strategy]['prediction'] for result in test_results]
    strategy_predictions_matrix.append(strategy_preds)

im = plt.imshow(strategy_predictions_matrix, cmap='RdYlBu_r', aspect='auto')
plt.colorbar(im)
plt.title('Strategy Predictions Heatmap')
plt.ylabel('Fusion Strategies')
plt.xlabel('Test Cases')
plt.yticks(range(len(strategies)), strategies)
plt.xticks(range(len(test_case_names)), [name[:10] + '...' for name in test_case_names], rotation=45)

# Plot 5: Confidence vs Prediction Scatter
plt.subplot(2, 3, 5)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i, strategy in enumerate(strategies):
    preds = [result['prediction'] for result in analysis_data[strategy]]
    confs = [result['confidence'] for result in analysis_data[strategy]]
    plt.scatter(preds, confs, label=strategy, color=colors[i], alpha=0.7, s=60)

plt.title('Confidence vs Prediction')
plt.xlabel('Prediction Score')
plt.ylabel('Confidence Score')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

# Plot 6: Strategy Agreement Analysis
plt.subplot(2, 3, 6)
agreement_scores = []
for result in test_results:
    predictions = [result['results'][strategy]['prediction'] for strategy in strategies]
    agreement = 1.0 - np.std(predictions)  # Higher agreement = lower std
    agreement_scores.append(agreement)

plt.plot(range(1, len(agreement_scores) + 1), agreement_scores, 
         marker='o', linewidth=2, markersize=8, color='#2ca02c')
plt.title('Strategy Agreement Across Test Cases')
plt.xlabel('Test Case Number')
plt.ylabel('Agreement Score')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(agreement_scores) + 1))

plt.tight_layout()
plt.show()

# Save fusion strategy configuration
config_data = {
    'fusion_strategies': fusion_engine.fusion_strategies,
    'confidence_thresholds': fusion_engine.confidence_thresholds,
    'test_results': test_results,
    'strategy_statistics': strategy_stats,
    'timestamp': datetime.now().isoformat()
}

# Save configuration
os.makedirs('phase7_fusion_configs', exist_ok=True)
config_file = 'phase7_fusion_configs/fusion_strategy_config.json'
with open(config_file, 'w') as f:
    json.dump(config_data, f, indent=2, default=str)

print(f"\nFusion strategy configuration saved to: {config_file}")

print("\n" + "="*80)
print("TASK 1 STATUS: FUSION STRATEGY DEVELOPMENT COMPLETED")
print("DELIVERABLE: CNN + Rule-based combination algorithms implemented")
print("NEXT: Task 2 - Weighted voting and confidence scoring")
print("="*80)


In [ ]:
# PHASE 7 - TASK 1B: WEIGHTED VOTING AND CONFIDENCE SCORING
# Advanced weighted voting mechanisms with dynamic confidence scoring

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PHASE 7 - TASK 1B: WEIGHTED VOTING AND CONFIDENCE SCORING")
print("="*80)
print("Task: Advanced weighted voting mechanisms with dynamic confidence assessment")
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print()

class WeightedVotingEngine:
    """
    Advanced Weighted Voting Engine with Dynamic Confidence Scoring
    Implements multiple voting strategies with confidence-based weight adjustments
    """
    
    def __init__(self):
        self.voting_methods = {}
        self.confidence_functions = {}
        self.performance_history = []
        self.initialize_voting_methods()
        self.initialize_confidence_functions()
        
    def initialize_voting_methods(self):
        """Initialize different weighted voting methods"""
        
        print("1. INITIALIZING WEIGHTED VOTING METHODS:")
        print("-" * 55)
        
        # Method 1: Static Performance-Based Weighting
        self.voting_methods['performance_based'] = {
            'name': 'Performance-Based Static Weighting',
            'description': 'Weights based on historical accuracy (CNN: 99.51%, Rule: 84.86%)',
            'weights': {
                'cnn': 0.7263,      # 99.51 / (99.51 + 84.86)
                'rule_based': 0.2737 # 84.86 / (99.51 + 84.86)
            },
            'weight_type': 'static'
        }
        
        # Method 2: Confidence-Weighted Dynamic Voting
        self.voting_methods['confidence_weighted'] = {
            'name': 'Confidence-Weighted Dynamic Voting',
            'description': 'Weights adjust based on individual prediction confidence',
            'base_weights': {'cnn': 0.65, 'rule_based': 0.35},
            'confidence_multiplier': 1.5,
            'weight_type': 'dynamic'
        }
        
        # Method 3: Exponential Confidence Weighting
        self.voting_methods['exponential_confidence'] = {
            'name': 'Exponential Confidence Weighting',
            'description': 'Exponential scaling of weights based on confidence',
            'base_weights': {'cnn': 0.6, 'rule_based': 0.4},
            'confidence_exponent': 2.0,
            'weight_type': 'exponential'
        }
        
        # Method 4: Adaptive Threshold Voting
        self.voting_methods['adaptive_threshold'] = {
            'name': 'Adaptive Threshold Voting',
            'description': 'Dynamic threshold adjustment based on confidence levels',
            'base_threshold': 0.5,
            'confidence_threshold_low': 0.4,
            'confidence_threshold_high': 0.8,
            'weight_type': 'threshold_adaptive'
        }
        
        # Method 5: Ensemble Confidence Voting
        self.voting_methods['ensemble_confidence'] = {
            'name': 'Ensemble Confidence Voting',
            'description': 'Meta-learning approach combining multiple voting methods',
            'meta_weights': [0.3, 0.25, 0.25, 0.2],
            'weight_type': 'ensemble'
        }
        
        for method_key, method in self.voting_methods.items():
            print(f"  Method: {method['name']:<35}")
            print(f"          {method['description']}")
            print(f"          Type: {method['weight_type']}")
            print()
        
        print(f"Total Voting Methods: {len(self.voting_methods)}")
        
    def initialize_confidence_functions(self):
        """Initialize confidence scoring functions"""
        
        print("\n2. INITIALIZING CONFIDENCE SCORING FUNCTIONS:")
        print("-" * 55)
        
        # Confidence Function 1: Sigmoid-based Confidence
        self.confidence_functions['sigmoid'] = {
            'name': 'Sigmoid Confidence Scoring',
            'description': 'Sigmoid transformation for smooth confidence curves',
            'steepness': 8.0,
            'midpoint': 0.5
        }
        
        # Confidence Function 2: Entropy-based Confidence
        self.confidence_functions['entropy'] = {
            'name': 'Entropy-based Confidence',
            'description': 'Information entropy for uncertainty quantification',
            'base_entropy': 1.0
        }
        
        # Confidence Function 3: Distance-based Confidence
        self.confidence_functions['distance'] = {
            'name': 'Distance-based Confidence',
            'description': 'Distance from decision boundary (0.5)',
            'boundary': 0.5,
            'scaling_factor': 2.0
        }
        
        # Confidence Function 4: Variance-based Confidence
        self.confidence_functions['variance'] = {
            'name': 'Prediction Variance Confidence',
            'description': 'Confidence based on prediction variance',
            'variance_threshold': 0.1
        }
        
        for conf_key, conf_func in self.confidence_functions.items():
            print(f"  Function: {conf_func['name']:<25}")
            print(f"            {conf_func['description']}")
            print()
            
        print(f"Total Confidence Functions: {len(self.confidence_functions)}")
    
    def calculate_sigmoid_confidence(self, prediction):
        """Calculate sigmoid-based confidence score"""
        params = self.confidence_functions['sigmoid']
        steepness = params['steepness']
        midpoint = params['midpoint']
        
        # Distance from midpoint
        distance = abs(prediction - midpoint)
        
        # Sigmoid transformation
        confidence = 1 / (1 + np.exp(-steepness * distance))
        return confidence
    
    def calculate_entropy_confidence(self, prediction):
        """Calculate entropy-based confidence score"""
        # Avoid log(0) by adding small epsilon
        epsilon = 1e-8
        p = max(epsilon, min(1 - epsilon, prediction))
        
        # Binary entropy
        entropy = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
        
        # Convert entropy to confidence (lower entropy = higher confidence)
        max_entropy = 1.0
        confidence = 1 - (entropy / max_entropy)
        return confidence
    
    def calculate_distance_confidence(self, prediction):
        """Calculate distance-based confidence score"""
        params = self.confidence_functions['distance']
        boundary = params['boundary']
        scaling = params['scaling_factor']
        
        # Distance from decision boundary
        distance = abs(prediction - boundary)
        
        # Scale to confidence score
        confidence = min(1.0, distance * scaling)
        return confidence
    
    def calculate_variance_confidence(self, predictions_list):
        """Calculate variance-based confidence for multiple predictions"""
        if len(predictions_list) < 2:
            return 0.5  # Default confidence for single prediction
            
        variance = np.var(predictions_list)
        threshold = self.confidence_functions['variance']['variance_threshold']
        
        # Lower variance = higher confidence
        confidence = max(0.1, 1 - (variance / threshold))
        return min(1.0, confidence)
    
    def performance_based_voting(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """Method 1: Performance-based static weighting"""
        weights = self.voting_methods['performance_based']['weights']
        
        final_prediction = (cnn_pred * weights['cnn']) + (rule_pred * weights['rule_based'])
        final_confidence = (cnn_conf * weights['cnn']) + (rule_conf * weights['rule_based'])
        
        return {
            'prediction': final_prediction,
            'confidence': final_confidence,
            'method': 'performance_based',
            'weights_used': weights,
            'details': {
                'static_weights': True,
                'cnn_contribution': cnn_pred * weights['cnn'],
                'rule_contribution': rule_pred * weights['rule_based']
            }
        }
    
    def confidence_weighted_voting(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """Method 2: Confidence-weighted dynamic voting"""
        base_weights = self.voting_methods['confidence_weighted']['base_weights']
        multiplier = self.voting_methods['confidence_weighted']['confidence_multiplier']
        
        # Adjust weights based on confidence
        conf_total = cnn_conf + rule_conf
        if conf_total > 0:
            cnn_boost = (cnn_conf / conf_total) * multiplier
            rule_boost = (rule_conf / conf_total) * multiplier
            
            # Dynamic weight calculation
            cnn_weight = base_weights['cnn'] + (cnn_boost - 1.0) * 0.1
            rule_weight = base_weights['rule_based'] + (rule_boost - 1.0) * 0.1
            
            # Normalize weights
            weight_sum = cnn_weight + rule_weight
            cnn_weight /= weight_sum
            rule_weight /= weight_sum
        else:
            cnn_weight = base_weights['cnn']
            rule_weight = base_weights['rule_based']
        
        final_prediction = (cnn_pred * cnn_weight) + (rule_pred * rule_weight)
        final_confidence = (cnn_conf * cnn_weight) + (rule_conf * rule_weight)
        
        return {
            'prediction': final_prediction,
            'confidence': final_confidence,
            'method': 'confidence_weighted',
            'weights_used': {'cnn': cnn_weight, 'rule_based': rule_weight},
            'details': {
                'base_weights': base_weights,
                'confidence_adjustment': {
                    'cnn_boost': cnn_boost if conf_total > 0 else 1.0,
                    'rule_boost': rule_boost if conf_total > 0 else 1.0
                }
            }
        }
    
    def exponential_confidence_voting(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """Method 3: Exponential confidence weighting"""
        base_weights = self.voting_methods['exponential_confidence']['base_weights']
        exponent = self.voting_methods['exponential_confidence']['confidence_exponent']
        
        # Exponential confidence scaling
        cnn_exp_conf = np.power(cnn_conf, exponent)
        rule_exp_conf = np.power(rule_conf, exponent)
        
        # Calculate exponential weights
        exp_total = cnn_exp_conf + rule_exp_conf
        if exp_total > 0:
            cnn_weight = (base_weights['cnn'] + cnn_exp_conf / exp_total) / 2
            rule_weight = (base_weights['rule_based'] + rule_exp_conf / exp_total) / 2
            
            # Normalize
            weight_sum = cnn_weight + rule_weight
            cnn_weight /= weight_sum
            rule_weight /= weight_sum
        else:
            cnn_weight = base_weights['cnn']
            rule_weight = base_weights['rule_based']
        
        final_prediction = (cnn_pred * cnn_weight) + (rule_pred * rule_weight)
        final_confidence = (cnn_conf * cnn_weight) + (rule_conf * rule_weight)
        
        return {
            'prediction': final_prediction,
            'confidence': final_confidence,
            'method': 'exponential_confidence',
            'weights_used': {'cnn': cnn_weight, 'rule_based': rule_weight},
            'details': {
                'exponential_confidences': {
                    'cnn': cnn_exp_conf,
                    'rule_based': rule_exp_conf
                },
                'exponent_used': exponent
            }
        }
    
    def adaptive_threshold_voting(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """Method 4: Adaptive threshold voting"""
        params = self.voting_methods['adaptive_threshold']
        base_threshold = params['base_threshold']
        conf_low = params['confidence_threshold_low']
        conf_high = params['confidence_threshold_high']
        
        # Calculate average confidence
        avg_confidence = (cnn_conf + rule_conf) / 2
        
        # Adapt threshold based on confidence
        if avg_confidence < conf_low:
            # Low confidence: more conservative (higher threshold)
            adapted_threshold = base_threshold + 0.2
        elif avg_confidence > conf_high:
            # High confidence: more aggressive (lower threshold)
            adapted_threshold = base_threshold - 0.1
        else:
            # Medium confidence: use base threshold
            adapted_threshold = base_threshold
        
        # Weighted prediction
        avg_prediction = (cnn_pred + rule_pred) / 2
        
        # Binary decision with adapted threshold
        binary_decision = 1 if avg_prediction > adapted_threshold else 0
        
        # Confidence adjustment based on threshold adaptation
        threshold_confidence = avg_confidence * (1 - abs(avg_prediction - adapted_threshold))
        
        return {
            'prediction': binary_decision,
            'confidence': threshold_confidence,
            'method': 'adaptive_threshold',
            'weights_used': {'cnn': 0.5, 'rule_based': 0.5},  # Equal weights for threshold method
            'details': {
                'base_threshold': base_threshold,
                'adapted_threshold': adapted_threshold,
                'average_prediction': avg_prediction,
                'threshold_adjustment': adapted_threshold - base_threshold
            }
        }
    
    def ensemble_confidence_voting(self, cnn_pred, rule_pred, cnn_conf, rule_conf):
        """Method 5: Ensemble confidence voting"""
        meta_weights = self.voting_methods['ensemble_confidence']['meta_weights']
        
        # Get results from other methods
        results = []
        results.append(self.performance_based_voting(cnn_pred, rule_pred, cnn_conf, rule_conf))
        results.append(self.confidence_weighted_voting(cnn_pred, rule_pred, cnn_conf, rule_conf))
        results.append(self.exponential_confidence_voting(cnn_pred, rule_pred, cnn_conf, rule_conf))
        results.append(self.adaptive_threshold_voting(cnn_pred, rule_pred, cnn_conf, rule_conf))
        
        # Extract predictions and confidences
        predictions = [r['prediction'] for r in results]
        confidences = [r['confidence'] for r in results]
        
        # Meta-weighted ensemble
        ensemble_prediction = sum(pred * weight for pred, weight in zip(predictions, meta_weights))
        ensemble_confidence = sum(conf * weight for conf, weight in zip(confidences, meta_weights))
        
        # Calculate ensemble agreement
        prediction_variance = np.var(predictions)
        confidence_variance = np.var(confidences)
        
        return {
            'prediction': ensemble_prediction,
            'confidence': ensemble_confidence,
            'method': 'ensemble_confidence',
            'weights_used': {'meta_weights': meta_weights},
            'details': {
                'individual_predictions': predictions,
                'individual_confidences': confidences,
                'prediction_variance': prediction_variance,
                'confidence_variance': confidence_variance,
                'ensemble_agreement': 1 - prediction_variance  # Higher agreement = lower variance
            }
        }
    
    def comprehensive_voting_test(self):
        """Test all voting methods with comprehensive scenarios"""
        
        print("\n3. COMPREHENSIVE WEIGHTED VOTING TEST:")
        print("-" * 55)
        
        # Comprehensive test scenarios
        test_scenarios = [
            {
                'name': 'High CNN Confidence - Clear Malicious',
                'cnn_pred': 0.95, 'cnn_conf': 0.92,
                'rule_pred': 0.78, 'rule_conf': 0.71,
                'ground_truth': 1,
                'scenario_type': 'high_cnn_confidence'
            },
            {
                'name': 'High Rule Confidence - Clear Malicious',
                'cnn_pred': 0.67, 'cnn_conf': 0.58,
                'rule_pred': 0.91, 'rule_conf': 0.89,
                'ground_truth': 1,
                'scenario_type': 'high_rule_confidence'
            },
            {
                'name': 'Both High Confidence - Agreement',
                'cnn_pred': 0.88, 'cnn_conf': 0.85,
                'rule_pred': 0.84, 'rule_conf': 0.82,
                'ground_truth': 1,
                'scenario_type': 'high_agreement'
            },
            {
                'name': 'Clear Disagreement - CNN vs Rule',
                'cnn_pred': 0.25, 'cnn_conf': 0.72,
                'rule_pred': 0.85, 'rule_conf': 0.76,
                'ground_truth': 0,  # Actually benign, rule-based wrong
                'scenario_type': 'clear_disagreement'
            },
            {
                'name': 'Low Confidence - Uncertain Case',
                'cnn_pred': 0.52, 'cnn_conf': 0.48,
                'rule_pred': 0.49, 'rule_conf': 0.43,
                'ground_truth': 0,
                'scenario_type': 'low_confidence'
            },
            {
                'name': 'Edge Case - Boundary Decision',
                'cnn_pred': 0.501, 'cnn_conf': 0.55,
                'rule_pred': 0.499, 'rule_conf': 0.53,
                'ground_truth': 1,
                'scenario_type': 'edge_case'
            },
            {
                'name': 'High CNN, Low Rule - Complementary',
                'cnn_pred': 0.89, 'cnn_conf': 0.87,
                'rule_pred': 0.34, 'rule_conf': 0.41,
                'ground_truth': 1,
                'scenario_type': 'complementary'
            },
            {
                'name': 'Clean Benign - Both Agree',
                'cnn_pred': 0.15, 'cnn_conf': 0.83,
                'rule_pred': 0.22, 'rule_conf': 0.79,
                'ground_truth': 0,
                'scenario_type': 'clean_benign'
            }
        ]
        
        # Test all voting methods
        voting_methods = [
            ('performance_based', self.performance_based_voting),
            ('confidence_weighted', self.confidence_weighted_voting),
            ('exponential_confidence', self.exponential_confidence_voting),
            ('adaptive_threshold', self.adaptive_threshold_voting),
            ('ensemble_confidence', self.ensemble_confidence_voting)
        ]
        
        comprehensive_results = []
        
        for i, scenario in enumerate(test_scenarios, 1):
            print(f"\nScenario {i}: {scenario['name']}")
            print(f"CNN: pred={scenario['cnn_pred']:.3f}, conf={scenario['cnn_conf']:.3f}")
            print(f"Rule: pred={scenario['rule_pred']:.3f}, conf={scenario['rule_conf']:.3f}")
            print(f"Ground Truth: {'Malicious' if scenario['ground_truth'] == 1 else 'Benign'}")
            print()
            
            scenario_results = {'scenario': scenario, 'method_results': {}}
            
            for method_name, method_func in voting_methods:
                result = method_func(
                    scenario['cnn_pred'], scenario['rule_pred'],
                    scenario['cnn_conf'], scenario['rule_conf']
                )
                
                # Calculate accuracy for this scenario
                binary_pred = 1 if result['prediction'] > 0.5 else 0
                correct = binary_pred == scenario['ground_truth']
                
                result['binary_prediction'] = binary_pred
                result['correct'] = correct
                result['accuracy'] = 1.0 if correct else 0.0
                
                scenario_results['method_results'][method_name] = result
                
                print(f"  {method_name:20}: pred={result['prediction']:.3f}, "
                      f"conf={result['confidence']:.3f}, "
                      f"binary={'Mal' if binary_pred == 1 else 'Ben'}, "
                      f"{'CORRECT' if correct else 'WRONG'}")
            
            comprehensive_results.append(scenario_results)
            print("-" * 50)
        
        return comprehensive_results

# Initialize and test the weighted voting engine
print("INITIALIZING WEIGHTED VOTING ENGINE...")
voting_engine = WeightedVotingEngine()

# Run comprehensive voting tests
comprehensive_test_results = voting_engine.comprehensive_voting_test()

print("\n4. VOTING METHOD PERFORMANCE ANALYSIS:")
print("-" * 55)

# Analyze performance across all scenarios
method_names = ['performance_based', 'confidence_weighted', 'exponential_confidence', 
                'adaptive_threshold', 'ensemble_confidence']

method_performance = {method: {'correct': 0, 'total': 0, 'confidences': [], 'predictions': []} 
                     for method in method_names}

for result in comprehensive_test_results:
    for method_name in method_names:
        method_result = result['method_results'][method_name]
        method_performance[method_name]['total'] += 1
        if method_result['correct']:
            method_performance[method_name]['correct'] += 1
        method_performance[method_name]['confidences'].append(method_result['confidence'])
        method_performance[method_name]['predictions'].append(method_result['prediction'])

# Calculate performance metrics
for method_name in method_names:
    perf = method_performance[method_name]
    accuracy = perf['correct'] / perf['total']
    avg_confidence = np.mean(perf['confidences'])
    confidence_std = np.std(perf['confidences'])
    
    print(f"{method_name.upper()}:")
    print(f"  Accuracy: {accuracy:.3f} ({perf['correct']}/{perf['total']})")
    print(f"  Avg Confidence: {avg_confidence:.3f} (±{confidence_std:.3f})")
    print()

# Create comprehensive visualizations
print("5. GENERATING WEIGHTED VOTING VISUALIZATIONS:")
print("-" * 55)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Method Accuracy Comparison
accuracies = [method_performance[method]['correct'] / method_performance[method]['total'] 
              for method in method_names]
axes[0,0].bar(method_names, accuracies, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
axes[0,0].set_title('Voting Method Accuracy Comparison')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_xticklabels(method_names, rotation=45)
axes[0,0].grid(True, alpha=0.3)
axes[0,0].set_ylim(0, 1)

# Plot 2: Average Confidence by Method
avg_confidences = [np.mean(method_performance[method]['confidences']) for method in method_names]
axes[0,1].bar(method_names, avg_confidences, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
axes[0,1].set_title('Average Confidence by Method')
axes[0,1].set_ylabel('Average Confidence')
axes[0,1].set_xticklabels(method_names, rotation=45)
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Confidence Distribution
for i, method in enumerate(method_names):
    confidences = method_performance[method]['confidences']
    axes[0,2].hist(confidences, alpha=0.6, label=method, bins=10)
axes[0,2].set_title('Confidence Score Distributions')
axes[0,2].set_xlabel('Confidence Score')
axes[0,2].set_ylabel('Frequency')
axes[0,2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0,2].grid(True, alpha=0.3)

# Plot 4: Scenario Performance Heatmap
scenario_names = [result['scenario']['name'][:15] + '...' for result in comprehensive_test_results]
performance_matrix = []
for method in method_names:
    method_accuracies = [result['method_results'][method]['accuracy'] 
                        for result in comprehensive_test_results]
    performance_matrix.append(method_accuracies)

im = axes[1,0].imshow(performance_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1,0].set_title('Method Performance by Scenario')
axes[1,0].set_ylabel('Voting Methods')
axes[1,0].set_xlabel('Test Scenarios')
axes[1,0].set_yticks(range(len(method_names)))
axes[1,0].set_yticklabels(method_names)
axes[1,0].set_xticks(range(len(scenario_names)))
axes[1,0].set_xticklabels(scenario_names, rotation=45, ha='right')
plt.colorbar(im, ax=axes[1,0])

# Plot 5: Prediction vs Confidence Scatter
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i, method in enumerate(method_names):
    preds = method_performance[method]['predictions']
    confs = method_performance[method]['confidences']
    axes[1,1].scatter(preds, confs, label=method, color=colors[i], alpha=0.7, s=60)

axes[1,1].set_title('Prediction vs Confidence by Method')
axes[1,1].set_xlabel('Prediction Score')
axes[1,1].set_ylabel('Confidence Score')
axes[1,1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1,1].grid(True, alpha=0.3)

# Plot 6: Method Agreement Analysis
agreement_scores = []
for result in comprehensive_test_results:
    predictions = [result['method_results'][method]['prediction'] for method in method_names]
    agreement = 1.0 - np.std(predictions)
    agreement_scores.append(agreement)

axes[1,2].plot(range(1, len(agreement_scores) + 1), agreement_scores, 
               marker='o', linewidth=2, markersize=8, color='#2ca02c')
axes[1,2].set_title('Method Agreement Across Scenarios')
axes[1,2].set_xlabel('Scenario Number')
axes[1,2].set_ylabel('Agreement Score')
axes[1,2].grid(True, alpha=0.3)
axes[1,2].set_xticks(range(1, len(agreement_scores) + 1))

plt.tight_layout()
plt.show()

# Save weighted voting configuration
voting_config = {
    'voting_methods': voting_engine.voting_methods,
    'confidence_functions': voting_engine.confidence_functions,
    'comprehensive_test_results': comprehensive_test_results,
    'method_performance': {
        method: {
            'accuracy': perf['correct'] / perf['total'],
            'total_tests': perf['total'],
            'correct_predictions': perf['correct'],
            'avg_confidence': float(np.mean(perf['confidences'])),
            'confidence_std': float(np.std(perf['confidences']))
        }
        for method, perf in method_performance.items()
    },
    'timestamp': datetime.now().isoformat()
}

# Save configuration
import os
os.makedirs('phase7_fusion_configs', exist_ok=True)
config_file = 'phase7_fusion_configs/weighted_voting_config.json'
with open(config_file, 'w') as f:
    json.dump(voting_config, f, indent=2, default=str)

print(f"Weighted voting configuration saved to: {config_file}")

print("\n" + "="*80)
print("TASK 1B STATUS: WEIGHTED VOTING AND CONFIDENCE SCORING COMPLETED")
print("DELIVERABLE: Advanced weighted voting mechanisms with dynamic confidence")
print("NEXT: Task 1c - Decision fusion optimization")
print("="*80)
